# Hands-on 1: Precipitation Forecast Post-Processing

**Module 2 — Deep Learning Best Practices & Applications**  
CIMA Foundation PhD Course, 27 May 2026

In this notebook we build a simple CNN to post-process (bias-correct) NWP precipitation forecasts.  
We use the **Modern ML Stack**: Zarr + xarray for data, PyTorch Lightning for training, Hydra/OmegaConf for configuration, and MLFlow for experiment tracking.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meteocima/Monaco_DLCourse/blob/main/notebooks/01_precipitation_postprocessing.ipynb)

## 0. Setup

We use [**uv**](https://docs.astral.sh/uv/) as our Python package manager — it's a fast, modern replacement for pip + virtualenv.

| Context | Command | What it does |
|---------|---------|-------------|
| **New project** | `uv init` | Scaffold a project with `pyproject.toml` |
| **Add a dependency** | `uv add torch xarray` | Add packages to `pyproject.toml` and install them |
| **Install everything** | `uv sync` | Create a `.venv` and install from the lockfile |
| **Run a command** | `uv run jupyter lab` | Run inside the managed `.venv` |
| **Colab / system Python** | `uv pip install --system -r requirements.txt` | Install without a `.venv` (for environments you don't manage) |

On **Colab** there is no `.venv` — Google provides the Python environment — so we use `uv pip install --system`.

In [ ]:
import os

# On Colab: clone the repo (or pull latest) and install dependencies
if "COLAB_RELEASE_TAG" in os.environ:
    if os.path.exists("/content/Monaco_DLCourse"):
        !git -C /content/Monaco_DLCourse pull -q
    else:
        !git clone https://github.com/meteocima/Monaco_DLCourse.git /content/Monaco_DLCourse
    %cd /content/Monaco_DLCourse/notebooks
    !pip install -q uv
    !uv pip install --system -q -r ../requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import zarr
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from omegaconf import OmegaConf
import mlflow

print(f"PyTorch version: {torch.__version__}")
print(f"Lightning version: {pl.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "\n\u26a0\ufe0f  No GPU detected!\n"
        "   Go to Runtime \u2192 Change runtime type \u2192 T4 GPU\n"
        "   Then re-run this cell.\n"
        "   Training will still work on CPU, but will be slower."
    )

## 1. Configuration with Hydra / OmegaConf

In production, Hydra loads YAML config files and composes them automatically via its `@hydra.main` decorator. In a notebook we use **OmegaConf** (Hydra's config backend) to load the same YAML files directly — giving us structured, typed configuration without needing the CLI.

The config lives in `configs/precipitation.yaml`. You can override any value inline after loading.

In [ ]:
# Load config from YAML (same file Hydra would use in a script)
cfg = OmegaConf.load("../configs/precipitation.yaml")

# You can override values inline, e.g.:
# cfg.training.max_epochs = 50
# cfg.data.batch_size = 64

print(OmegaConf.to_yaml(cfg))

## 2. Data Loading with Zarr + xarray

We load data from the [WeatherBench 2](https://weatherbench2.readthedocs.io/) cloud Zarr stores (no download needed!):
- **Truth**: ERA5 reanalysis precipitation
- **Forecast**: Real ECMWF IFS HRES operational forecasts at **all 6-hourly lead times** up to `max_lead_time`

Each (initialisation time, lead time) pair becomes one sample. The **lead time** is fed to the CNN as a second input channel (a constant field normalised to [0, 1]), so the model can learn lead-time-dependent corrections.

In [ ]:
# Open the WeatherBench 2 Zarr stores (lazy — no data downloaded yet)
# token="anon" forces anonymous access, avoiding Colab credential issues
gcs_opts = {"token": "anon"}
ds_era5 = xr.open_zarr(cfg.data.era5_zarr_url, storage_options=gcs_opts)
ds_hres = xr.open_zarr(cfg.data.hres_zarr_url, storage_options=gcs_opts, decode_timedelta=True)

print("ERA5 variables:", list(ds_era5.data_vars))
print("HRES variables:", list(ds_hres.data_vars))
print(f"\nHRES lead times: {ds_hres.prediction_timedelta.values[[0, -1]]}")

In [ ]:
import pandas as pd

max_lead = pd.Timedelta(cfg.data.max_lead_time)
steps = pd.timedelta_range("6h", max_lead, freq="6h")
print(f"Lead-time steps ({len(steps)}): {steps[0]}  …  {steps[-1]}")

# Select HRES forecasts for all lead-time steps
fc_raw = (
    ds_hres[cfg.data.variable]
    .sel(prediction_timedelta=steps)
    .sel(time=slice(cfg.data.time_start, cfg.data.time_end))
    .isel(latitude=slice(0, cfg.data.lat_slice))
    .transpose("time", "prediction_timedelta", "latitude", "longitude")
)
print(f"HRES shape: {dict(fc_raw.sizes)}")

init_times = fc_raw.time.values
n_init = len(init_times)
n_steps = len(steps)

# Compute valid times for every (init, step) pair
valid_times_2d = init_times[:, None] + steps.values[None, :]  # (n_init, n_steps)

# Load ERA5 truth at those valid times (may contain duplicates — xarray handles that)
truth_raw = (
    ds_era5[cfg.data.variable]
    .sel(time=valid_times_2d.ravel())
    .isel(latitude=slice(0, cfg.data.lat_slice))
    .transpose("time", "latitude", "longitude")
)

# Save coordinates for plotting
lats = fc_raw.latitude.values
lons = fc_raw.longitude.values

# Reshape to flat sample dimension: (n_init * n_steps, lat, lon)
forecast = fc_raw.values.astype(np.float32).reshape(-1, len(lats), len(lons))
truth = truth_raw.values.astype(np.float32)

# Build normalised lead-time array (0 → 1) for each sample
step_hours = steps.total_seconds() / 3600
max_hours = max_lead.total_seconds() / 3600
lead_norm = np.tile((step_hours / max_hours).values.astype(np.float32), n_init)

print(f"\nSamples: {len(forecast)}  ({n_init} init times × {n_steps} steps)")
print(f"Forecast: {forecast.shape}, Truth: {truth.shape}, Lead norm: {lead_norm.shape}")
print(f"Mean bias (forecast − truth): {(forecast - truth).mean():.4f} mm")

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Show a sample from the last lead-time step of the first init time
idx = n_steps - 1  # last step of first init
sample_lead_h = step_hours[idx % n_steps]
proj = ccrs.PlateCarree()
lon2d, lat2d = np.meshgrid(lons, lats)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), subplot_kw={"projection": proj})

titles = ["IFS HRES Forecast", "ERA5 Truth", "Forecast Error"]
data = [forecast[idx], truth[idx], forecast[idx] - truth[idx]]
cmaps = ["Blues", "Blues", "RdBu_r"]
vmin_max = [(0, None), (0, None), (None, None)]

for ax, title, d, cmap, (vn, vx) in zip(axes, titles, data, cmaps, vmin_max):
    im = ax.pcolormesh(lon2d, lat2d, d, cmap=cmap, vmin=vn, vmax=vx, transform=proj)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle="--")
    ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, shrink=0.8)

fig.suptitle(f"6h Precipitation — {sample_lead_h:.0f}h lead time (mm)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. PyTorch Dataset & DataModule

In [ ]:
class PrecipitationDataset(Dataset):
    """Dataset for precipitation post-processing with lead-time channel."""

    def __init__(self, forecast, truth, lead_norm):
        self.forecast = torch.from_numpy(forecast).unsqueeze(1)  # (N, 1, H, W)
        self.truth = torch.from_numpy(truth).unsqueeze(1)
        self.lead_norm = torch.from_numpy(lead_norm).float()     # (N,)

    def __len__(self):
        return len(self.forecast)

    def __getitem__(self, idx):
        fc = self.forecast[idx]                                   # (1, H, W)
        lead_ch = self.lead_norm[idx].expand_as(fc)               # (1, H, W)
        x = torch.cat([fc, lead_ch], dim=0)                       # (2, H, W)
        return x, self.truth[idx]


class PrecipDataModule(pl.LightningDataModule):
    """Lightning DataModule for precipitation data."""

    def __init__(self, forecast, truth, lead_norm, cfg):
        super().__init__()
        self.forecast = forecast
        self.truth = truth
        self.lead_norm = lead_norm
        self.cfg = cfg

    def setup(self, stage=None):
        n = len(self.forecast)
        n_train = int(n * self.cfg.data.train_split)
        n_val = int(n * self.cfg.data.val_split)
        self.train_ds = PrecipitationDataset(
            self.forecast[:n_train], self.truth[:n_train], self.lead_norm[:n_train]
        )
        self.val_ds = PrecipitationDataset(
            self.forecast[n_train:n_train + n_val],
            self.truth[n_train:n_train + n_val],
            self.lead_norm[n_train:n_train + n_val],
        )
        self.test_ds = PrecipitationDataset(
            self.forecast[n_train + n_val:],
            self.truth[n_train + n_val:],
            self.lead_norm[n_train + n_val:],
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds, batch_size=self.cfg.data.batch_size, shuffle=True
        )

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.cfg.data.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.cfg.data.batch_size)


dm = PrecipDataModule(forecast, truth, lead_norm, cfg)
dm.setup()
print(f"Train: {len(dm.train_ds)}, Val: {len(dm.val_ds)}, Test: {len(dm.test_ds)}")

## 4. Model: Simple CNN for Post-Processing

A residual CNN that learns to correct the forecast. The input has **two channels**:
1. The precipitation forecast field
2. A constant field encoding the normalised lead time (0 → 1)

The model predicts a *correction field* (1 channel) that is added to the forecast channel via a residual connection.

In [ ]:
class PrecipPostProcessor(pl.LightningModule):
    """CNN that learns a lead-time-aware residual correction for precipitation forecasts."""

    def __init__(self, cfg):
        super().__init__()
        self.save_hyperparameters()
        self.cfg = cfg

        layers = []
        in_ch = cfg.model.in_channels
        for i in range(cfg.model.n_layers):
            out_ch = cfg.model.hidden_channels if i < cfg.model.n_layers - 1 else cfg.model.out_channels
            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1))
            if i < cfg.model.n_layers - 1:
                layers.append(nn.ReLU())
            in_ch = out_ch
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        correction = self.net(x)                # (B, 1, H, W)
        return x[:, :1] + correction            # residual on forecast channel only

    def training_step(self, batch, batch_idx):
        x, truth = batch
        pred = self(x)
        loss = F.mse_loss(pred, truth)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, truth = batch
        pred = self(x)
        loss = F.mse_loss(pred, truth)
        mae = F.l1_loss(pred, truth)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mae", mae)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.cfg.training.learning_rate)


model = PrecipPostProcessor(cfg)
print(model)

## 5. Training with Lightning + MLFlow Tracking

In [ ]:
# Set up MLFlow experiment
mlflow.set_experiment(cfg.mlflow.experiment_name)

with mlflow.start_run(run_name="cnn_postprocessor_v1"):
    # Log configuration
    mlflow.log_params(OmegaConf.to_container(cfg, resolve=True))

    # Create trainer
    trainer = pl.Trainer(
        max_epochs=cfg.training.max_epochs,
        accelerator=cfg.training.accelerator,
        enable_progress_bar=True,
        log_every_n_steps=5,
    )

    # Train
    trainer.fit(model, dm)

    # Log final metrics
    val_loss = trainer.callback_metrics.get("val_loss")
    val_mae = trainer.callback_metrics.get("val_mae")
    if val_loss is not None:
        mlflow.log_metric("final_val_loss", val_loss.item())
    if val_mae is not None:
        mlflow.log_metric("final_val_mae", val_mae.item())

    print(f"\nFinal val_loss: {val_loss}")
    print(f"Final val_mae: {val_mae}")

## 6. Evaluation & Visualization

In [ ]:
# Run predictions on test set
model.eval()
test_x = torch.stack([dm.test_ds[i][0] for i in range(len(dm.test_ds))])  # (N, 2, H, W)
test_truth = dm.test_ds.truth

with torch.no_grad():
    test_pred = model(test_x)

# Convert to numpy for plotting
test_forecast_np = test_x[:, 0].numpy()         # forecast channel only
test_truth_np = test_truth.squeeze(1).numpy()
test_pred_np = test_pred.squeeze(1).numpy()
test_lead_np = dm.test_ds.lead_norm.numpy()      # for per-lead-time analysis

In [ ]:
# Compare: raw forecast vs post-processed vs truth (test set)
proj = ccrs.PlateCarree()
fig, axes = plt.subplots(2, 3, figsize=(18, 10), subplot_kw={"projection": proj})

for row, idx in enumerate([0, n_steps - 1]):
    lead_h = test_lead_np[idx] * max_hours
    vmin = min(test_truth_np[idx].min(), test_forecast_np[idx].min())
    vmax = max(test_truth_np[idx].max(), test_forecast_np[idx].max())

    panels = [
        (f"HRES Forecast ({lead_h:.0f}h)", test_forecast_np[idx]),
        (f"Post-Processed ({lead_h:.0f}h)", test_pred_np[idx]),
        ("ERA5 Truth", test_truth_np[idx]),
    ]
    for col, (title, d) in enumerate(panels):
        ax = axes[row, col]
        im = ax.pcolormesh(lon2d, lat2d, d, cmap="Blues", vmin=vmin, vmax=vmax, transform=proj)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle="--")
        ax.gridlines(draw_labels=(row == 1), linewidth=0.3, alpha=0.5)
        ax.set_title(title)
        plt.colorbar(im, ax=ax, orientation="horizontal", pad=0.05, shrink=0.8)

plt.suptitle("Post-Processing Results (Test Set)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Per-lead-time MSE breakdown
unique_leads = np.unique(test_lead_np)
lead_hours = unique_leads * max_hours
raw_mses, pp_mses = [], []

for lv in unique_leads:
    mask = test_lead_np == lv
    raw_mses.append(np.mean((test_forecast_np[mask] - test_truth_np[mask]) ** 2))
    pp_mses.append(np.mean((test_pred_np[mask] - test_truth_np[mask]) ** 2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lead_hours, raw_mses, "o-", label="Raw HRES")
ax.plot(lead_hours, pp_mses, "s-", label="Post-processed (CNN)")
ax.set_xlabel("Lead time (hours)")
ax.set_ylabel("MSE")
ax.set_title("Post-processing skill vs lead time (test set)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Overall numbers
raw_mse = np.mean((test_forecast_np - test_truth_np) ** 2)
pp_mse = np.mean((test_pred_np - test_truth_np) ** 2)
print(f"Raw HRES forecast MSE:  {raw_mse:.4f}")
print(f"Post-processed MSE:     {pp_mse:.4f}")
print(f"Improvement:            {(1 - pp_mse/raw_mse) * 100:.1f}%")

## 7. Exercises

1. **Increase the horizon**: Try `max_lead_time: "5 days"` or `"10 days"` — how does the per-lead-time skill curve change?
2. **Change the architecture**: Try adding more layers or using a U-Net structure with skip connections
3. **Compare optimizers**: Try `SGD`, `AdamW`, or add a learning rate scheduler
4. **Add more input channels**: Load additional HRES variables from the Zarr store (e.g., `2m_temperature`, `geopotential`) as extra CNN input channels
5. **Try a different region**: Change `lat_slice` to select different latitude bands and see how precipitation patterns differ
6. **Check MLFlow**: Run `!mlflow ui` and open the link to explore logged experiments